# Load required packages

In [ ]:
from enum import unique

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
#import pooch
from scipy.sparse import csr_matrix
from scipy.io import mmwrite
from pathlib import Path
import anndata
import copy
from scipy.io import mmread
from scipy.sparse import csr_matrix
import glob
import re
print("scanpy version:", sc.__version__)
print("numpy version:", np.__version__)

In [ ]:
os.getcwd()

In [ ]:
os.chdir('/mnt/d/lxk/project/lishan-20260613/')
from helper.annotation_helper import *

In [ ]:
sc.settings.set_figure_params(dpi=300, dpi_save=500)
setGrid(True)

# Load and prepare data

In [ ]:
sampleInfo = [
    "NP1",
    "NP2",
    "NP3",
    "PE1",
    "PE2",
    "PE3",

]
sampleList = list()
print(Path.cwd())
for sample in sampleInfo:

    obsfile = glob.glob(f"./data/scRNA/{sample}/" + "/*barcodes.tsv.gz")[0]
    varfile = glob.glob(f"./data/scRNA/{sample}/" + "/*features.tsv.gz")[0]
    mtxfile = glob.glob(f"./data/scRNA/{sample}/" + "/*matrix.mtx.gz")[0]
    print(obsfile + " "+ varfile+ " " + mtxfile)

    obs = pd.read_csv(obsfile,header=None,index_col=0,sep = '\t')
    var = pd.read_csv(varfile,header=None,index_col=0,sep = '\t')
    if sample == 'NP3':
        var = pd.read_csv(varfile,header=None,index_col=1,sep = '\t')



    mtx = mmread(mtxfile)
    mtx = mtx.T
    mtx = csr_matrix(mtx)

    #修改标签
    var.index.name = None
    obs.index.name = None
    obs["sample"]  = sample
    import  re
    obs['group']  = re.sub(r"\d+", "", sample)
    adata = sc.AnnData(mtx,obs = obs,var = var)
    adata.var_names_make_unique()
    print(adata.shape)
    sampleList.append(copy.deepcopy(adata))
    del adata

#adatas = anndata.concat(sampleList,join = 'inner')
adatas = anndata.concat(sampleList, join='inner', fill_value=0)
print(adatas.shape)
adatas.obs_names_make_unique()
print(adatas.shape)
adatas.obs['sample'].value_counts()


In [ ]:
adatas.obs['sample'].value_counts()

In [ ]:
sc.pp.filter_genes(adatas, min_cells=3)
sc.pp.filter_cells(adatas, min_genes=200)

# Human gene symbols
adatas.var["mt"] = adatas.var_names.str.match(r"^MT-", case=False)
adatas.var["ribo"] = adatas.var_names.str.match(r"^RP[SL]", case=False)
adatas.var["hb"] = adatas.var_names.str.match(r"^HB(?!P)", case=False)

sc.pp.calculate_qc_metrics(
    adatas,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=None,
    log1p=False,
    inplace=True
)

sc.pl.violin(
    adatas,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_hb", "pct_counts_ribo"],
    jitter=0.4,
    multi_panel=True
)
#sc.pp.scrublet(adatas, batch_key="sample") #pip install scikit-image

In [ ]:

#过滤
adatas = adatas[adatas.obs.n_genes_by_counts < 6000, :]
adatas = adatas[adatas.obs.n_genes_by_counts > 300, :]

adatas = adatas[adatas.obs.total_counts < 50000, :]
#adatas = adatas[adatas.obs.predicted_doublet == False, :]

adatas = adatas[adatas.obs.pct_counts_mt < 7, :]
adatas = adatas[adatas.obs.pct_counts_hb < 3, :]
#adatas = adatas[adatas.obs.pct_counts_ribo < 40, :]
sc.pl.violin(adatas, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)


adatas.layers['counts'] = adatas.X.copy()


In [ ]:
adatas.X = adatas.layers["counts"].copy()
sc.experimental.pp.highly_variable_genes(
    adatas,
    flavor='pearson_residuals',
    n_top_genes=3000,
    layer='counts',
    batch_key='sample'
)
sc.pp.normalize_total(adatas,target_sum=1e4)
sc.pp.log1p(adatas)

adatas.layers['normalize'] = adatas.X.copy()

# SCVI

In [ ]:
import scvi  # 变分自编码器框架（scVI/scANVI 模型）
import torch  # 底层深度学习后端 scvi需要用到GPU
print("device_count:", torch.cuda.device_count())
adata_hvg = adatas[:, adatas.var.highly_variable].copy()
scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30, gene_likelihood="nb")
model.train()
SCVI_LATENT_KEY = "X_scVI"
adatas.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

sc.pp.neighbors(adatas, use_rep=SCVI_LATENT_KEY,n_neighbors=15)
sc.tl.umap(adatas)

# leiden

In [ ]:
import importlib
import helper.annotation_helper as ah
import helper.plot_helper as ph
importlib.reload(ah)
importlib.reload(ph)


In [ ]:
import helper.interactive_umap as iu

In [ ]:
iu.interactive_delete_cells(adatas)

In [ ]:
sc.tl.embedding_density(adatas, groupby="group")
sc.pl.embedding_density(adatas,
                        groupby="group",
                        bg_dotsize=5,
                        fg_dotsize=5,
                        color_map = "GnBu",
                        #cbar_kws={"shrink": 0.3},
                        )

In [ ]:
for resolution in [0.3,0.5,0.7,0.9,1.1]:
    sc.tl.leiden(adatas,
                 key_added='clusters',
                 flavor="igraph",
                 n_iterations=-1,
                 resolution = resolution,
                 )
    sc.pl.umap(
            adatas,
            color='clusters',
            legend_loc = 'on data'
        )

In [ ]:
sc.tl.leiden(adatas,
                key_added='clusters',
                flavor="igraph",
                n_iterations=-1,#-1为直到收敛为最佳聚类，-2是默认值，迭代两次
                resolution = 0.5,#默认1，越高分的越多
                )
sc.pl.umap(
        adatas,
        color='clusters',
        legend_loc = 'on data'
    )

In [ ]:
adatas.write('./h5ad/adatas_0.h5ad')

# annotation

In [ ]:
adatas = sc.read('./h5ad/adatas_0.h5ad')

In [ ]:
sc.tl.leiden(adatas,
                key_added='clusters',
                flavor="igraph",
                n_iterations=-1,#-1为直到收敛为最佳聚类，-2是默认值，迭代两次
                resolution = 0.7,#默认1，越高分的越多
                )
sc.pl.umap(
        adatas,
        color='clusters',
        legend_loc = 'on data'
    )

In [ ]:
sc.pl.umap(
        adatas,
        color='clusters',
        legend_loc = 'on data'
    )

In [ ]:
sc.tl.rank_genes_groups(
    adatas,
    groupby="clusters",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    layer="normalize" if "normalize" in adatas.layers else None
)

from scanpy.get import rank_genes_groups_df

df = rank_genes_groups_df(adatas, None)

df["pct_diff"] = df["pct_nz_group"] - df["pct_nz_reference"]

df = df[
    (df["logfoldchanges"] > 1) &
    (df["pct_nz_group"] > 0.4) &
    (df["pct_diff"] > 0.15) &
    (df["pvals_adj"] < 0.05)
]

df = df[
    ~df["names"].str.startswith(("MT-", "RPS", "RPL", "HB"))
]

df = df[[
    "group", "names", "scores", "logfoldchanges",
    "pvals_adj", "pct_nz_group", "pct_nz_reference", "pct_diff"
]]

df.to_csv("./export/all_marker.csv", index=False)

In [ ]:
marker_genes_dict = {
    "DSC": ["PRL", "IGFBP1", "RBP1"],
    "Fibroblast": ["COL1A1", "COL3A1", "DLK1"],
    "EVT": ["HLA-G", "MMP11", "AOC1"],
    "Trophoblast": ["KRT7", "PEG10", "PAGE4"],
    "Endo": ["PECAM1", "VWF", "CLDN5"],
    "Myeloid": ["LYZ", "HLA-DRA", "C1QA"],
    "T": ["CD3D","CD3E", "TRAC" ],
    "NK": ["NKG7", "GNLY", "KLRD1"],
}

sc.pl.dotplot(
    adatas,
    marker_genes_dict,
    groupby="clusters",
    dendrogram=False,
    cmap="YlGnBu", # https://matplotlib.org/stable/users/explain/colors/colormaps.html
    #dot_max=0.6,     # 点最大直径比例（默认 1.0）
    #dot_min=0.1,     # 点最小直径比例（默认 0）
    vmax=5,          # 显示的最大表达值，>3 的都显示为同一深红
    vmin=0,           # 最小值（默认0）
    #save="_markers.pdf"
    standard_scale="var"


)

In [ ]:
cluster2annotation = {
    "0": "DSC",
    "1": "EVT",
    "2": "Fibroblast",
    "3": "DSC",
    "4": "T",
    "5": "Trophoblast",
    "6": "Endo",
    "7": "Myeloid",
    "8": "NK",
    "9": "T",
    "10": "Endo",
    "11": "Myeloid",
    "12": "Endo",
}

adatas.obs["celltype"] = (
    adatas.obs["clusters"]
    .astype(str)
    .map(cluster2annotation)
)

In [ ]:
sc.pl.umap(
        adatas,
        color=['group','celltype','sample'],
        #legend_loc = 'on data'
    )

In [77]:

ordered_samples = ["NP1","NP2","NP3","PE1","PE2","PE3",]
ordered_groups = ['NP','PE']
ordered_celltype = list(marker_genes_dict.keys())

adatas.obs["celltype"] = pd.Categorical(
    adatas.obs["celltype"],
    categories=ordered_celltype,
    ordered=True,
)


#固定样本顺序

adatas.obs["sample"] = (
    adatas.obs["sample"]
    .astype("category")
    .cat.set_categories(ordered_samples, ordered=True)
)


adatas.obs["group"] = (
    adatas.obs["group"]
    .astype("category")
    .cat.set_categories(ordered_groups, ordered=True)
)

In [78]:
celltype_colors  = {
    "DSC": "#4C5F8F",
    "Fibroblast": "#8A6F4D",
    "EVT": "#D95F02",
    "Trophoblast": "#E7A6A1",
    "Endo": "#5BA8A0",
    "Myeloid": "#B95F62",
    "T": "#6FA083",
    "NK": "#8DA6BF",
}

cats = adatas.obs["celltype"].cat.categories
adatas.uns["celltype_colors"] = [celltype_colors[x] for x in cats]

## 新增一列 celltype_id

In [ ]:
celltype_order = list(celltype_colors.keys())

celltype2id = {
    celltype: i
    for i, celltype in enumerate(celltype_order)
}

adatas.obs["celltype_id"] = (
    adatas.obs["celltype"]
    .astype(str)
    .map(celltype2id)
)

adatas.obs["celltype_id"] = adatas.obs["celltype_id"].astype("category")

In [ ]:
print(celltype_colors.values())
ah.show_colors(celltype_colors.values())

In [79]:
group_colors = {
    "NP": "#8DA6BF",
    "PE": "#E7A6A1",
}

sample_colors = {
    "NP1": "#4C5F8F",
    "NP2": "#6D8BA6",
    "NP3": "#C58A54",
    "PE1": "#D8B56D",
    "PE2": "#6FA083",
    "PE3": "#9BAE6A",
}

cats = adatas.obs["group"].cat.categories
adatas.uns["group_colors"] = [group_colors[x] for x in cats]

cats = adatas.obs["sample"].cat.categories
adatas.uns["sample_colors"] = [sample_colors[x] for x in cats]

In [81]:
adatas.write_h5ad('./h5ad/rna_final.h5ad')